# ConvNeXt Tiny Unfreeze Comparison (Colab)

This Colab notebook is a lighter version of the local experiment notebook. It is meant for continuation testing on Google Colab, especially for deeper unfreezing settings such as 120 layers.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# Set allocator before importing TensorFlow to reduce fragmentation.
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# Update these paths for your Drive layout.
DATA_DIR = '/content/drive/MyDrive/DDSM/ROI'
OUTPUT_ROOT = '/content/drive/MyDrive/SW_training_outputs_colab'
CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Colab test defaults. Start conservative for deep unfreezing.
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
SEED = 123
UNFREEZE_CANDIDATES = [120]
STAGE1_EPOCHS = 10
STAGE2_EPOCHS = 25

assert os.path.exists(DATA_DIR), f'Dataset path not found: {DATA_DIR}'
print('DATA_DIR =', DATA_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)


In [ ]:
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, roc_auc_score
from tensorflow import keras

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision enabled:', mixed_precision.global_policy())
except Exception as exc:
    print('Mixed precision not enabled:', exc)

AUTOTUNE = tf.data.AUTOTUNE
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

print('TF version:', tf.__version__)
print('GPU devices:', gpus)
!nvidia-smi


In [ ]:
def count_files_by_class(root_dir):
    counts = {}
    for class_name in sorted(os.listdir(root_dir)):
        class_dir = os.path.join(root_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        counts[class_name] = sum(
            1 for file_name in os.listdir(class_dir)
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
        )
    return counts

def summarize_labels(ds, class_names):
    counts = Counter()
    for _, labels in ds:
        labels = labels.numpy().astype('int32').reshape(-1)
        counts.update(labels.tolist())
    return {class_names[idx]: counts.get(idx, 0) for idx in range(len(class_names))}

def prepare(ds, shuffle=False):
    if shuffle:
        ds = ds.shuffle(512, seed=SEED, reshuffle_each_iteration=True)
    return ds.prefetch(AUTOTUNE)

def collect_predictions(ds, model):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        pb = model.predict(xb, verbose=0).reshape(-1)
        y_true.append(yb.numpy().reshape(-1))
        y_prob.append(pb)
    return np.concatenate(y_true).astype('int32'), np.concatenate(y_prob)

print('Raw file counts by class:', count_files_by_class(DATA_DIR))

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='binary',
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='binary',
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

class_names = train_ds.class_names
temp_card = tf.data.experimental.cardinality(temp_ds).numpy()
val_batches = max(1, temp_card // 2)
test_batches = temp_card - val_batches
if test_batches == 0:
    raise ValueError('Test split is empty. Reduce batch size or use more data.')

val_ds = temp_ds.take(val_batches)
test_ds = temp_ds.skip(val_batches)
train_ds_prep = prepare(train_ds, shuffle=True)
val_ds_prep = prepare(val_ds)
test_ds_prep = prepare(test_ds)

print('Classes:', class_names)
print('Train batches:', tf.data.experimental.cardinality(train_ds).numpy())
print('Validation batches:', tf.data.experimental.cardinality(val_ds).numpy())
print('Test batches:', tf.data.experimental.cardinality(test_ds).numpy())
print('Train label counts:', summarize_labels(train_ds, class_names))
print('Validation label counts:', summarize_labels(val_ds, class_names))
print('Test label counts:', summarize_labels(test_ds, class_names))


In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.10),
    keras.layers.RandomZoom(0.20),
    keras.layers.RandomTranslation(0.05, 0.05),
    keras.layers.RandomContrast(0.20),
], name='data_augmentation')

def build_convnext_model():
    backbone = keras.applications.ConvNeXtTiny(
        include_top=False,
        include_preprocessing=False,
        weights='imagenet',
        input_shape=IMG_SIZE + (3,),
        pooling='avg',
        name='convnext_tiny',
    )
    preprocessing = keras.Sequential([
        keras.layers.Rescaling(1.0 / 255),
    ], name='preprocessing')
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = preprocessing(inputs)
    x = data_augmentation(x)
    x = backbone(x, training=False)
    x = keras.layers.Dense(256, activation='gelu')(x)
    x = keras.layers.Dropout(0.4)(x)
    outputs = keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='cancer_prob')(x)
    return keras.Model(inputs, outputs)

stage1_ckpt = os.path.join(CHECKPOINT_DIR, 'best_convnext_stage1_colab.keras')

if os.path.exists(stage1_ckpt):
    print('Loading existing Stage 1 checkpoint:', stage1_ckpt)
    model = keras.models.load_model(stage1_ckpt, compile=False)
else:
    print('Training Stage 1 from scratch.')
    model = build_convnext_model()
    backbone = model.get_layer('convnext_tiny')
    backbone.trainable = False
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.AUC(name='auc'),
            keras.metrics.BinaryAccuracy(name='acc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ],
    )
    cb1 = [
        keras.callbacks.ModelCheckpoint(
            stage1_ckpt,
            monitor='val_auc',
            mode='max',
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_auc',
            mode='max',
            patience=3,
            restore_best_weights=True,
            verbose=1,
        ),
    ]
    model.fit(
        train_ds_prep,
        validation_data=val_ds_prep,
        epochs=STAGE1_EPOCHS,
        callbacks=cb1,
        verbose=1,
    )
    keras.backend.clear_session()


In [ ]:
comparison_results = []
comparison_histories = {}

for num_layers_to_unfreeze in UNFREEZE_CANDIDATES:
    print(f'\n===== Stage 2 experiment: unfreeze last {num_layers_to_unfreeze} layers =====')
    keras.backend.clear_session()
    stage2_model = keras.models.load_model(stage1_ckpt, compile=False)
    stage2_backbone = stage2_model.get_layer('convnext_tiny')

    stage2_backbone.trainable = True
    for layer in stage2_backbone.layers:
        layer.trainable = False

    depth = min(num_layers_to_unfreeze, len(stage2_backbone.layers))
    for layer in stage2_backbone.layers[-depth:]:
        layer.trainable = True

    print('ConvNeXt backbone layers:', len(stage2_backbone.layers))
    print('Unfrozen last layers:', depth)
    print('Stage 2 trainable weights:', len(stage2_model.trainable_weights))

    stage2_model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=1e-4),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.AUC(name='auc'),
            keras.metrics.BinaryAccuracy(name='acc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ],
    )

    ckpt_path = os.path.join(CHECKPOINT_DIR, f'best_convnext_unfreeze_{depth}_colab.keras')
    cb2 = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor='val_auc',
            mode='max',
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_auc',
            mode='max',
            patience=5,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc',
            mode='max',
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    history2 = stage2_model.fit(
        train_ds_prep,
        validation_data=val_ds_prep,
        epochs=STAGE2_EPOCHS,
        callbacks=cb2,
        verbose=1,
    )
    comparison_histories[depth] = history2

    best_model = keras.models.load_model(ckpt_path, compile=False)
    best_model.compile(
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.AUC(name='auc'),
            keras.metrics.BinaryAccuracy(name='acc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ],
    )

    test_metrics = best_model.evaluate(test_ds_prep, return_dict=True, verbose=1)
    test_true, test_prob = collect_predictions(test_ds_prep, best_model)
    test_pred = (test_prob >= 0.5).astype('int32')

    result = {
        'unfreeze_layers': depth,
        'ckpt_path': ckpt_path,
        'best_val_auc': float(max(history2.history['val_auc'])),
        'test_auc': float(roc_auc_score(test_true, test_prob)),
        'test_acc': float(accuracy_score(test_true, test_pred)),
        'test_loss': float(test_metrics['loss']),
        'test_precision': float(test_metrics['precision']),
        'test_recall': float(test_metrics['recall']),
    }
    comparison_results.append(result)
    print('Result:', result)

comparison_results = sorted(comparison_results, key=lambda item: item['test_auc'], reverse=True)
print('\nBest setting by test AUC:', comparison_results[0])


In [ ]:
print('ConvNeXt unfreeze comparison summary (sorted by test AUC):')
for result in comparison_results:
    print(
        f"unfreeze={result['unfreeze_layers']:>3} | "
        f"val_auc={result['best_val_auc']:.4f} | "
        f"test_auc={result['test_auc']:.4f} | "
        f"test_acc={result['test_acc']:.4f} | "
        f"precision={result['test_precision']:.4f} | "
        f"recall={result['test_recall']:.4f}"
    )

if comparison_histories:
    plt.figure(figsize=(6, 4))
    for depth in sorted(comparison_histories):
        history = comparison_histories[depth].history
        plt.plot(history['val_auc'], marker='o', label=f'Val AUC {depth}')
    plt.title('Validation AUC by Unfreeze Depth')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
